# V11 — Clean, patient-disjoint split from the original Kermany release
**Study design v2, §3.1.** CPU only, ~10 min. No GPU or internet needed.

**What this notebook does**
1. Finds the original Kermany ChestXRay2017 tree (Kaggle mirror `paultimothymooney/chest-xray-pneumonia`),
   ignoring the nested duplicate copy and `__MACOSX`.
2. Extracts patient keys from filenames and hashes every image (MD5 + 16×16 dHash).
3. Groups images that share a patient key, an MD5, or a near-identical dHash (Hamming ≤ 4), using union-find.
4. **Test** = official Kermany test set (labels checked by a third expert) minus every group that also touches official train.
   **Val** = about 15 % of official-train groups (StratifiedGroupKFold, seed 42). **Train** = the rest.
5. Keeps one image per duplicate cluster inside each split. Nothing augmented is ever written.
6. Asserts zero patient / MD5 / dHash overlap across splits and writes `split_v2.csv` plus fingerprints.

Downstream notebooks (V12–V14) verify the fingerprints printed at the end before running.

In [ ]:
import os, re, json, hashlib, random, glob
import numpy as np, pandas as pd
from PIL import Image
from sklearn.model_selection import StratifiedGroupKFold
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

SEED = 42
DHASH_THR = 4          # Hamming distance on 256-bit dHash counted as near-duplicate
VAL_FOLDS = 7          # 1/7 ≈ 14.3 % of official-train groups go to validation
OUT = os.environ.get("V11_OUT", "/kaggle/working")
KERMANY_ROOT = os.environ.get("KERMANY_ROOT", "")      # optional override (local smoke test)
V1_AUDIT_CSV = os.environ.get("V1_AUDIT_CSV", "")      # optional: v1 audit_files.csv for the contrast table
os.makedirs(OUT, exist_ok=True)
random.seed(SEED); np.random.seed(SEED)

## 1. Locate the dataset

In [ ]:
def find_kermany_root():
    if KERMANY_ROOT:
        return KERMANY_ROOT
    cands = []
    for base in glob.glob("/kaggle/input/**/", recursive=True):
        if "__MACOSX" in base:
            continue
        if all(os.path.isdir(os.path.join(base, s, c)) for s in ("train", "test") for c in ("NORMAL", "PNEUMONIA")):
            cands.append(base.rstrip("/"))
    if not cands:
        raise FileNotFoundError("Kermany tree with train/test × NORMAL/PNEUMONIA not found under /kaggle/input")
    cands.sort(key=len)          # the mirror ships chest_xray/ and chest_xray/chest_xray/; take the shallowest
    return cands[0]

ROOT = find_kermany_root()
print("Kermany root:", ROOT)

IMG_EXT = (".jpeg", ".jpg", ".png")
rows = []
for off_split in ("train", "val", "test"):
    for label in ("NORMAL", "PNEUMONIA"):
        d = os.path.join(ROOT, off_split, label)
        if not os.path.isdir(d):
            continue
        for fn in sorted(os.listdir(d)):
            if fn.lower().endswith(IMG_EXT) and not fn.startswith("."):
                rows.append(dict(relpath=f"{off_split}/{label}/{fn}", official_split=off_split, label=label, fname=fn))
df = pd.DataFrame(rows)
# The official val folder (16 images) belongs to the official training pool
df["pool"] = np.where(df.official_split == "test", "official_test", "official_train")
print(df.groupby(["official_split", "label"]).size().unstack(fill_value=0))
assert len(df) > 0

## 2. Patient keys
`personN_bacteria_M` → `person_N_bacteria`; `personN_virus_M` → `person_N_virus`; `IM-XXXX-YYYY` → `im_XXXX`;
`NORMAL2-IM-XXXX-YYYY` → `normal2_XXXX`. Keys are merged **across folders**.

**Why the pathogen is part of the pneumonia key.** Kermany numbers persons separately for bacterial and viral cases:
`person1_bacteria_*` and `person1_virus_*` are different children. A first run with a pathogen-free key purged 320 of 624
official test images (deviation log, study design §10). Section 2b below re-tests this on every run: repeats of the same
person + pathogen are far more similar than random pairs, whereas same-number / other-pathogen pairs are not.

In [ ]:
AUG_PAT = re.compile(r"_aug_\d+", re.I)
def patient_key(fn):
    s = fn.lower()
    m = re.match(r"person(\d+)_(bacteria|virus)", s)
    if m: return f"person_{int(m.group(1))}_{m.group(2)}"
    m = re.match(r"normal2-im-(\d+)-", s)
    if m: return f"normal2_{int(m.group(1))}"
    m = re.match(r"im-(\d+)-", s)
    if m: return f"im_{int(m.group(1))}"
    return f"unk_{s}"
df["patient_key"] = df.fname.map(patient_key)
df["has_aug_suffix"] = df.fname.str.contains(AUG_PAT)
print("unparsed keys:", int(df.patient_key.str.startswith("unk_").sum()), "| files with _aug_ suffix:", int(df.has_aug_suffix.sum()))
assert df.has_aug_suffix.sum() == 0, "original release must not contain augmented files"
kp = df.groupby("patient_key").pool.nunique()
print("patient keys present in BOTH official train and official test:", int((kp > 1).sum()), "(expected ≈ 0 with pathogen-specific keys)")

## 3. Content hashes (MD5 + 16×16 dHash)

In [ ]:
def dhash256(img):
    g = img.convert("L").resize((17, 16), Image.BILINEAR)
    a = np.asarray(g, dtype=np.int16)
    return np.packbits((a[:, 1:] > a[:, :-1]).flatten())     # 32 bytes

md5s, dh, sizes = [], [], []
for rp in df.relpath:
    p = os.path.join(ROOT, rp)
    with open(p, "rb") as fh:
        md5s.append(hashlib.md5(fh.read()).hexdigest())
    with Image.open(p) as im:
        dh.append(dhash256(im)); sizes.append(im.size)
df["md5"] = md5s
df["w"] = [s[0] for s in sizes]; df["h"] = [s[1] for s in sizes]
DH = np.stack(dh)
df["dhash"] = [x.tobytes().hex() for x in DH]
print("exact-duplicate files:", int(df.md5.duplicated(keep=False).sum()))

## 2b. Evidence check: is the person number shared across pathogens? (runs after hashing)

In [ ]:
def _namespace_test():
    POP = np.array([bin(i).count("1") for i in range(256)], dtype=np.uint8)
    pp = df[df.fname.str.lower().str.startswith("person")].reset_index(drop=True)
    pnum = pp.fname.str.extract(r"person(\d+)_", expand=False).astype(int)
    typ = pp.fname.str.extract(r"person\d+_(bacteria|virus)", expand=False)
    Hh = np.stack([np.frombuffer(bytes.fromhex(h), dtype=np.uint8) for h in pp.dhash])
    ham = lambda i, j: int(POP[Hh[i] ^ Hh[j]].sum())
    rng_ = np.random.default_rng(SEED)
    same = [(i, j) for idx in pp.groupby([pnum, typ]).indices.values() for a, i in enumerate(idx) for j in idx[a + 1:]]
    cross = []
    for _, d in pp.groupby(pnum):
        b = d.index[typ[d.index] == "bacteria"]; v = d.index[typ[d.index] == "virus"]
        cross += [(i, j) for i in b for j in v]
    rand = [tuple(rng_.choice(len(pp), 2, replace=False)) for _ in range(20000)]
    from scipy.stats import mannwhitneyu
    out, H_ = [], {}
    for name, pr in (("same person + pathogen", same), ("same number, other pathogen", cross), ("random pneumonia pairs", rand)):
        H_[name] = np.array([ham(i, j) for i, j in pr]) if pr else np.array([np.nan])
        wd = [abs(pp.w[i] - pp.w[j]) / max(pp.w[i], pp.w[j]) for i, j in pr] if pr else [np.nan]
        out.append(dict(pairs=name, n=len(pr), hamming_median=float(np.nanmedian(H_[name])), width_diff_median=float(np.nanmedian(wd))))
    t = pd.DataFrame(out)
    t["p_vs_random"] = [mannwhitneyu(H_[k], H_["random pneumonia pairs"]).pvalue if len(H_[k]) > 1 else np.nan for k in t.pairs]
    t.to_csv(os.path.join(OUT, "v11_key_namespace_test.csv"), index=False)
    print(t.round(4).to_string(index=False))
    return t
NS_TEST = _namespace_test()

## 4. Near-duplicate pairs (all pairs, blocked popcount)

In [ ]:
POP = np.array([bin(i).count("1") for i in range(256)], dtype=np.uint8)
N = len(df); pairs = []
for i0 in range(0, N, 256):
    blk = DH[i0:i0 + 256]
    ham = POP[np.bitwise_xor(blk[:, None, :], DH[None, :, :])].sum(-1, dtype=np.int32)
    ii, jj = np.nonzero(ham <= DHASH_THR)
    for a, b in zip(ii + i0, jj):
        if a < b:
            pairs.append((int(a), int(b), int(ham[a - i0, b])))
near = pd.DataFrame(pairs, columns=["i", "j", "hamming"])
near["cross_class"] = df.label.values[near.i] != df.label.values[near.j]
near["cross_pool"] = df.pool.values[near.i] != df.pool.values[near.j]
print(f"near-duplicate pairs (Hamming <= {DHASH_THR}): {len(near)} | cross-class {int(near.cross_class.sum())} | cross-pool {int(near.cross_pool.sum())}")
# Contact sheet of up to 50 random pairs for the visual check required by the design (§3.1)
if len(near):
    smp = near.sample(min(50, len(near)), random_state=SEED)
    fig, axs = plt.subplots(len(smp), 2, figsize=(3, 1.5 * len(smp)), squeeze=False)
    for r, (_, pr) in enumerate(smp.iterrows()):
        for c, k in enumerate((int(pr.i), int(pr.j))):
            with Image.open(os.path.join(ROOT, df.relpath.iloc[k])) as im:
                axs[r, c].imshow(im.convert("L").resize((128, 128)), cmap="gray")
            axs[r, c].set_axis_off()
            axs[r, c].set_title(f"{df.relpath.iloc[k][-28:]}  d={int(pr.hamming)}", fontsize=4)
    fig.tight_layout(); fig.savefig(os.path.join(OUT, "v11_near_duplicate_sample.png"), dpi=150); plt.close(fig)

## 5. Union-find grouping: patient key ∪ MD5 ∪ near-duplicate

In [ ]:
class DSU:
    def __init__(self, n): self.p = list(range(n))
    def find(self, a):
        while self.p[a] != a:
            self.p[a] = self.p[self.p[a]]; a = self.p[a]
        return a
    def union(self, a, b):
        ra, rb = self.find(a), self.find(b)
        if ra != rb: self.p[rb] = ra

grp = DSU(N); dup = DSU(N)          # grp: patient ∪ content;  dup: content only
for col, sets in (("patient_key", (grp,)), ("md5", (grp, dup))):
    for idx in df.groupby(col).indices.values():
        for k in idx[1:]:
            for s in sets: s.union(int(idx[0]), int(k))
for a, b in zip(near.i, near.j):
    grp.union(a, b); dup.union(a, b)
df["group"] = [grp.find(k) for k in range(N)]
df["dup_cluster"] = [dup.find(k) for k in range(N)]
g = df.groupby("group")
lab_conf = g.label.nunique() > 1
conf_groups = set(lab_conf[lab_conf].index)
print("groups:", g.ngroups, "| label-conflict groups dropped:", len(conf_groups),
      "images:", int(df.group.isin(conf_groups).sum()))

## 6. Split assignment

In [ ]:
df["split_v2"] = ""; df["drop_reason"] = ""
df.loc[df.group.isin(conf_groups), "drop_reason"] = "label_conflict"
pools_per_group = g.pool.nunique()
mixed_groups = set(pools_per_group[pools_per_group > 1].index) - conf_groups

is_test_pool = df.pool == "official_test"
purge = is_test_pool & df.group.isin(mixed_groups) & (df.drop_reason == "")
df.loc[purge, "drop_reason"] = "test_group_touches_train"
df.loc[is_test_pool & (df.drop_reason == ""), "split_v2"] = "test"
print(f"official-test images purged: {int(purge.sum())} of {int(is_test_pool.sum())} "
      f"({purge.sum() / max(int(is_test_pool.sum()), 1):.1%})")
if purge.sum() / max(int(is_test_pool.sum()), 1) > 0.30:
    print("WARNING: >30 % of the official test set purged; patient-key namespaces may restart per folder. "
          "Log this in the design deviation table (§10) before proceeding.")

tr = df[(df.pool == "official_train") & (df.drop_reason == "")]
sgkf = StratifiedGroupKFold(n_splits=VAL_FOLDS, shuffle=True, random_state=SEED)
_, val_idx = next(sgkf.split(tr, tr.label, groups=tr.group))
val_groups = set(tr.iloc[val_idx].group)
df.loc[tr.index, "split_v2"] = np.where(tr.group.isin(val_groups), "val", "train")

# One image per content-duplicate cluster inside each split (patient-only links are kept: different studies)
assigned = df[df.split_v2 != ""].sort_values("relpath")
keep_idx = assigned.drop_duplicates(["split_v2", "dup_cluster"]).index
redundant = assigned.index.difference(keep_idx)
df.loc[redundant, "drop_reason"] = "duplicate_within_split"; df.loc[redundant, "split_v2"] = ""
print(df[df.split_v2 != ""].groupby(["split_v2", "label"]).size().unstack(fill_value=0))
print("dropped:", df.drop_reason.replace("", np.nan).dropna().value_counts().to_dict())

## 7. Integrity assertions (the notebook stops if any fails)

In [ ]:
S = df[df.split_v2 != ""]
def overlap(col):
    sets = {s: set(S.loc[S.split_v2 == s, col]) for s in ("train", "val", "test")}
    return {f"{a}-{b}": len(sets[a] & sets[b]) for a, b in (("train", "val"), ("train", "test"), ("val", "test"))}
ov_pk, ov_md5, ov_grp = overlap("patient_key"), overlap("md5"), overlap("group")
sp = df.split_v2.values
nd_cross = int(sum(1 for a, b in zip(near.i, near.j) if sp[a] and sp[b] and sp[a] != sp[b]))
print("patient-key overlap:", ov_pk); print("MD5 overlap:", ov_md5); print("group overlap:", ov_grp)
print("near-duplicate pairs across splits:", nd_cross)
assert all(v == 0 for v in ov_pk.values()), "patient leakage"
assert all(v == 0 for v in ov_md5.values()), "exact-duplicate leakage"
assert all(v == 0 for v in ov_grp.values()), "group leakage"
assert nd_cross == 0, "near-duplicate leakage"
assert not S.has_aug_suffix.any()
assert set(S.loc[S.split_v2 == "test", "official_split"]) <= {"test"}, "test must contain official-test originals only"
assert S.md5.is_unique, "duplicate content inside a split"
print("ALL INTEGRITY CHECKS PASSED")

## 8. Fingerprints, outputs, audit figure

In [ ]:
def fingerprint(paths):
    return hashlib.md5("|".join(sorted(paths)).encode("utf-8")).hexdigest()[:16]
SP = ["train", "val", "test"]
FP = {s: fingerprint(S.loc[S.split_v2 == s, "relpath"]) for s in SP}
counts = {s: {k: int(v) for k, v in S[S.split_v2 == s].label.value_counts().items()} for s in SP}
summary = dict(
    root=ROOT, seed=SEED, dhash_threshold=DHASH_THR, n_files=int(N),
    official_counts={f"{a}/{b}": int(v) for (a, b), v in df.groupby(["official_split", "label"]).size().items()},
    exact_duplicate_files=int(df.md5.duplicated(keep=False).sum()),
    near_duplicate_pairs=int(len(near)), near_duplicate_cross_class=int(near.cross_class.sum()),
    near_duplicate_cross_pool=int(near.cross_pool.sum()),
    patient_keys=int(df.patient_key.nunique()), keys_in_both_official_pools=int((kp > 1).sum()),
    groups=int(g.ngroups), label_conflict_groups=len(conf_groups),
    test_purged=int(purge.sum()), test_official_total=int(is_test_pool.sum()),
    dropped={k: int(v) for k, v in df.drop_reason.replace("", np.nan).dropna().value_counts().items()},
    split_counts=counts, fingerprints=FP,
    overlaps=dict(patient_key=ov_pk, md5=ov_md5, group=ov_grp, near_duplicate_cross=nd_cross),
)
df.drop(columns=["dup_cluster"]).to_csv(os.path.join(OUT, "split_v2.csv"), index=False)
near.to_csv(os.path.join(OUT, "v11_near_duplicate_pairs.csv"), index=False)
with open(os.path.join(OUT, "audit_v2_summary.json"), "w") as fh:
    json.dump(summary, fh, indent=2)

contrast = [dict(dataset="v2 (this split)", augmented_files=0,
                 test_normals_sharing_source_with_train="0.0 % (asserted)",
                 exact_duplicate_groups_across_splits=0, patient_overlap_train_test=0)]
if V1_AUDIT_CSV and os.path.exists(V1_AUDIT_CSV):
    a = pd.read_csv(V1_AUDIT_CSV)
    dupg = a[a.md5.duplicated(keep=False)].groupby("md5").split.nunique()
    contrast.insert(0, dict(dataset="v1 balanced derivative", augmented_files=int(a["ad"].str.contains("_aug_").sum()),
                            test_normals_sharing_source_with_train="86.6 % (re-split test, v1 audit)",
                            exact_duplicate_groups_across_splits=int((dupg > 1).sum()),
                            patient_overlap_train_test="610 (v1 audit)"))
pd.DataFrame(contrast).to_csv(os.path.join(OUT, "v11_v1_vs_v2_leakage.csv"), index=False)

fig, axs = plt.subplots(1, 3, figsize=(7.16, 2.4), layout="constrained", gridspec_kw=dict(width_ratios=[1, 1.4, 1]))
for ax_ in axs:
    ax_.xaxis.set_major_locator(matplotlib.ticker.MaxNLocator(integer=True, nbins=4))
for j, lab in enumerate(("NORMAL", "PNEUMONIA")):
    axs[0].bar(np.arange(3) + (j - 0.5) * 0.38, [counts[s].get(lab, 0) for s in SP], 0.38,
               color=["#3D5A80", "#B04A1F"][j], label=lab.lower())
axs[0].set_xticks(range(3)); axs[0].set_xticklabels(SP); axs[0].set_ylabel("images")
axs[0].legend(frameon=False, fontsize=6); axs[0].set_title("Split v2, patient-disjoint", loc="left", fontsize=8)
dr = df.drop_reason.replace("", np.nan).dropna().value_counts()
if len(dr):
    axs[1].barh([s.replace("_", " ") for s in dr.index], dr.values, color="#5A5A5A")
axs[1].set_xlabel("images removed"); axs[1].set_title("Why images were removed", loc="left", fontsize=8)
if len(near):
    axs[2].hist(near.hamming, bins=np.arange(-0.5, DHASH_THR + 1.5), color="#5A5A5A")
axs[2].set_xlabel("Hamming distance"); axs[2].set_ylabel("pairs")
axs[2].set_title("Near-duplicate pairs", loc="left", fontsize=8)
axs[0].xaxis.set_major_locator(matplotlib.ticker.FixedLocator(range(3)))
fig.savefig(os.path.join(OUT, "fig_split_v2_audit.png"), dpi=300); plt.close(fig)

print("\n" + "=" * 60)
print("  SPLIT v2 FINGERPRINTS (paste into V12–V14):")
for s in SP:
    print(f"    {s:<5} {FP[s]}   {counts[s]}")
print("=" * 60)